## Installing Librairies

In [ ]:
!pip install azure-storage-blob
#!pip install boto3
#!pip install google-cloud-storage

## Importing Librairies

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from bs4 import BeautifulSoup
import os
import re
import requests
from azure.storage.blob import BlobServiceClient
import json

## Configuration

In [ ]:
def load_config(config_path="config.json"):
    """Load the Azure configuration parameters from the config.json file."""
    with open(config_path, "r", encoding="utf-8") as config_file:
        config = json.load(config_file)
    return config["AZURE_CONNECTION_STRING"], config["CONTAINER_NAME"]

AZURE_CONNECTION_STRING, CONTAINER_NAME = load_config()
print(f"AZURE_CONNECTION_STRING: {AZURE_CONNECTION_STRING}")
print(f"CONTAINER_NAME: {CONTAINER_NAME}")

## Data Gathering

In [14]:
def extract_parquet_links(html_file):
    """Read the HTML file and extract URLs that point to .parquet files."""
    with open(html_file, "r", encoding="utf-8") as file:
        html_content = file.read()
    soup = BeautifulSoup(html_content, "html.parser")
    # Find all <a> elements with an href attribute and filter for .parquet files
    links = soup.find_all("a", href=True)
    parquet_links = [link['href'].strip() for link in links if ".parquet" in link['href'].lower()]
    return parquet_links

def download_file(url, local_filename):
    """Download a file from URL and save it locally."""
    print(f"Downloading: {url}")
    response = requests.get(url, stream=True)
    response.raise_for_status()  # Raise an exception for HTTP errors
    with open(local_filename, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
    print(f"Downloaded to: {local_filename}")
    return local_filename

def extract_year_from_filename(filename):
    """
    Extracts a four-digit year from the filename using regex.
    Assumes filenames have a pattern like: 'yellow_tripdata_2025-01.parquet'
    """
    match = re.search(r"_(\d{4})-\d{2}\.parquet", filename)
    if match:
        return match.group(1)
    else:
        return "unknown"

def upload_to_azure(local_filename, blob_path):
    """
    Upload the local file to Azure Blob Storage.
    The blob_path defines the path in the container (simulated folders using slashes).
    """
    # Create the BlobServiceClient object
    blob_service_client = BlobServiceClient.from_connection_string(AZURE_CONNECTION_STRING)
    container_client = blob_service_client.get_container_client(CONTAINER_NAME)

    # Attempt to create the container if it doesn't already exist
    try:
        container_client.create_container()
    except Exception as e:
        # Likely the container already exists
        print(f"Container creation skipped: {e}")

    # Get a BlobClient using the desired blob_path
    blob_client = blob_service_client.get_blob_client(container=CONTAINER_NAME, blob=blob_path)
    print(f"Uploading {local_filename} to Azure blob: {blob_path}")

    # Upload the file (overwriting if it already exists)
    with open(local_filename, "rb") as data:
        blob_client.upload_blob(data, overwrite=True)
    print("Upload complete.")

In [15]:
def main():
    html_file = "taxidata.html"
    parquet_links = extract_parquet_links(html_file)
    print(f"Found {len(parquet_links)} parquet links.")

    for url in parquet_links:
        # Extract the filename from the URL (everything after the last "/")
        filename = url.split("/")[-1]
        # Download the file locally
        local_path = download_file(url, filename)
        # Extract the year from the filename
        year = extract_year_from_filename(filename)
        # Construct the target blob path: raw/{year}/{filename}
        blob_path = f"raw/{year}/{filename}"
        # Upload the file to Azure Blob Storage
        upload_to_azure(local_path, blob_path)
        # Optionally remove the local file after upload
        os.remove(local_path)

In [ ]:
main()